<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [5]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [3]:
mlflow.set_experiment(
    "assignment"
)

2026/05/17 18:56:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/17 18:56:57 INFO mlflow.store.db.utils: Updating database tables
2026/05/17 18:56:59 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/Matheus/Cesar/Machine_learning/atividade-03-mlp-Matheuslh/notebooks/mlruns/1', creation_time=1779055019571, experiment_id='1', last_update_time=1779055019571, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [4]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


def load_data(seed):
    X, y = fetch_openml('Fashion-MNIST', version=1, return_X_y=True, as_frame=False)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    return X_train, X_val, y_train, y_val


Sim, é necessário normalizar os dados para esse tipo de modelo. Pois o mapeamento das entradas para uma escala uniforme é o que garante a estabilidade e a velocidade do treinamento da rede. Sem esse ajuste, variáveis com amplitudes numéricas muito distintas distorcem as curvas de nível da função de custo, assim, gerando uma superfície extremamente alongada que faz com que algoritmos de otimização oscilem violentamente e demorem a convergir para o mínimo global. Além disso, entradas com valores de grande magnitude forçam os neurônios para as zonas de saturação de funções de ativação comuns, onde as derivadas são praticamente nulas.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [6]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed, max_iter=200, batch_size='auto'):
    
    model = MLPClassifier(
        activation=activation,
        hidden_layer_sizes=hidden_layers,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=max_iter,
        batch_size=batch_size
    )
    
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    
    return acc, prec, rec, f1

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [ ]:
import time


#Definindo os hiperparâmetros do experimento
activation = 'relu'
hidden_layers = (100, 50)
learning_rate = 0.001
max_iter = 200
batch_size = 128
seed = 42

#Executando o experimento
with mlflow.start_run(run_name="mlp_fashion_mnist"):
    
    #Treinamento
    start_time = time.time()
    
    seed = 42
    X_train, X_val, y_train, y_val = load_data(seed)
    
    model = train_mlp(
        X_train=X_train, 
        y_train=y_train, 
        activation=activation, 
        hidden_layers=hidden_layers, 
        learning_rate=learning_rate, 
        seed=seed,
        max_iter=max_iter,
        batch_size=batch_size
    )
    
    end_time = time.time()
    training_time = end_time - start_time
    
    #Avaliação
    accuracy, precision, recall, f1 = evaluate(model, X_val, y_val) 
    
    #Registrando Parâmetros
    mlflow.log_param("activation", activation)
    mlflow.log_param("hidden_layers", str(hidden_layers)) 
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("max_iter", max_iter)
    mlflow.log_param("batch_size", batch_size)
    
    #Registrando Métricas
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("training_time", training_time)
    
    
    print(f"Acurácia obtida: {accuracy:.4f} | Tempo: {training_time:.2f}s")

Experimento executado e registrado com sucesso no MLflow!
Acurácia obtida: 0.8734 | Tempo: 501.99s


# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [ ]:
# TODO: implemente

## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [ ]:
# TODO: implemente

## Responda:

- Redes maiores sempre melhoraram os resultados?
- Redes maiores sempre melhoraram os resultados?
- Qual arquitetura apresentou melhor tradeoff?

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [ ]:
# TODO: implemente

## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


# TODO: responda aqui